In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

import os
import json

results = []
if os.path.exists("../reports/results.json"):
    try:
        with open("../reports/results.json") as f:
            results = json.load(f)
    except (json.JSONDecodeError, ValueError):
        results = []

def log_run(results, entry):
    results = [r for r in results if r["run_name"] != entry["run_name"]]
    results.append(entry)
    return results

df = pd.read_parquet('../data/loans_clean.parquet')

df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df["credit_history_months"] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df["annual_inc"].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

df_train = df[df['issue_year'].isin([2014, 2015])].copy()
df_val   = df[df["issue_year"] == 2016].copy()
df_test  = df[df['issue_year'] == 2017].copy()

y_train = df_train['default'].values
y_val   = df_val['default'].values

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

Train: 598,647  |  Val: 293,095  |  Test: 169,300


In [2]:
gain_rate = 0.10
loss_rate = 0.50

def calculate_profit(df_subset, approval_mask):
    """Total profit from a set of approval decisions."""
    approved = df_subset[approval_mask]
    if len(approved) == 0:
        return 0
    profit = np.where(
        approved["default"] == 0,
         gain_rate * approved["loan_amnt"],
        -loss_rate * approved["loan_amnt"]
    )
    return profit.sum()

In [3]:
df_train_fe = df_train.copy()
df_val_fe = df_val.copy()
df_test_fe = df_test.copy()

def prep_features(df):
    """Apply feature engineering decisions from EDA."""
    df = df.copy()
    
    redundant = [
        "fico_range_high",
        "funded_amnt",
        "funded_amnt_inv",
        "num_sats",
        "installment",
        "num_rev_tl_bal_gt_0",
    ]

    joint_cols = [c for c in df.columns if c.startswith("sec_app") or c.endswith("_joint")]

    high_cardinality = ["zip_code", "sub_grade"]

    split_cols = ["issue_year"]

    emp_map = {
        "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
        "5 years": 5, "6 years": 6, '7 years': 7, "8 years": 8, "9 years": 9,
        "10+ years": 10
    }
    df["emp_length"] = df["emp_length"].map(emp_map)
    
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    
    df = df.drop(
        columns=[c for c in cols_to_drop if c in df.columns]
    )
    
    return df



df_train_fe = prep_features(df_train_fe)
df_val_fe = prep_features(df_val_fe)
df_test_fe = prep_features(df_test_fe)

print(f"Features after prep: {df_train_fe.shape[1]}")
print(f"\nTrain shape: {df_train_fe.shape}")
print(f"Val shape:   {df_val_fe.shape}")

Features after prep: 80

Train shape: (598647, 80)
Val shape:   (293095, 80)


In [4]:
y_train = df_train_fe["default"].values
y_val = df_val_fe["default"].values

x_train = df_train_fe.drop(columns=["default"])
x_val = df_val_fe.drop(columns=['default'])

x_train['term'] = x_train['term'].str.extract(r'(\d+)').astype(int)
x_val['term']   = x_val['term'].str.extract(r'(\d+)').astype(int)

categorical_cols = x_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = x_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumeric columns: {len(numeric_cols)} features")

Categorical columns (7): ['grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type']

Numeric columns: 72 features


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# numeric pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
    ]
)

# categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# both
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
    ]
) 

classifier = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='lbfgs',
        random_state=42
    ))
    ]
)

print("Pipeline built. Steps:")
for name, step in classifier.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

Pipeline built. Steps:
  preprocessor: ColumnTransformer
  classifier: LogisticRegression


In [6]:
from sklearn.ensemble import RandomForestClassifier
import time

rf_baseline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=20,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ))
])

print("Training Random Forest ")

start = time.time()

rf_baseline.fit(x_train, y_train)

train_time = time.time() - start

print(f"Done in {train_time:.1f}s")

val_probs = rf_baseline.predict_proba(x_val)[:, 1]
auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)
brier = brier_score_loss(y_val, val_probs)

thresholds = np.linspace(0.05, 0.55, 100)

profits = [
    calculate_profit(df_val, val_probs < t)
    for t in thresholds
]

best_idx = np.argmax(profits)

best_threshold = thresholds[best_idx]
best_profit = profits[best_idx]

approval_rate = (val_probs < best_threshold).mean()

results = log_run(results, {
    "run_name": "rf_baseline",
    "model_type": "RandomForestClassifier",
    "n_estimators": 200,
    "max_depth": 20,
    "min_samples_leaf": 20,
    "best_threshold": float(best_threshold),
    "auc": float(auc),
    "pr_auc": float(pr_auc),
    "brier": float(brier),
    "profit_at_threshold": float(best_profit),
    "approval_rate": float(approval_rate),
    "train_time_seconds": float(train_time),
    }
)

print(f"\n=== RANDOM FOREST (baseline) ===")
print(f"Train time:     {train_time:.1f}s")
print(f"AUC:            {auc:.4f}")
print(f"PR-AUC:         {pr_auc:.4f}")
print(f"Brier:          {brier:.4f}")
print(f"Best threshold: {best_threshold:.4f}")
print(f"Approval rate:  {approval_rate:.2%}")
print(f"Profit:         ${best_profit:,.0f}")

lr = next(r for r in results if r["run_name"] == "logreg_baseline")
print(f"\n--- Comparison to LogReg ---")
print(f"LogReg:    AUC {lr['auc']:.4f}, Profit ${lr['profit_at_threshold']/1e6:.1f}M")
print(f"RF:        AUC {auc:.4f}, Profit ${best_profit/1e6:.1f}M")
print(f"Delta:     AUC {auc - lr['auc']:+.4f}, Profit ${(best_profit - lr['profit_at_threshold'])/1e6:+.1f}M")

Training Random Forest 
Done in 41.9s

=== RANDOM FOREST (baseline) ===
Train time:     41.9s
AUC:            0.7159
PR-AUC:         0.4273
Brier:          0.2005
Best threshold: 0.3581
Approval rate:  36.26%
Profit:         $60,507,642

--- Comparison to LogReg ---
LogReg:    AUC 0.7157, Profit $60.7M
RF:        AUC 0.7159, Profit $60.5M
Delta:     AUC +0.0003, Profit $-0.2M


In [7]:
from sklearn.ensemble import RandomForestClassifier
import time

# Random forest without class balancing
rf_no_balance = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=20,
        class_weight=None,
        n_jobs=-1,
        random_state=42
    ))
])

print("Training RF (no class balancing)...")

start = time.time()

rf_no_balance.fit(x_train, y_train)

train_time = time.time() - start

val_probs = rf_no_balance.predict_proba(x_val)[:, 1]

auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)
brier = brier_score_loss(y_val, val_probs)

thresholds = np.linspace(0.05, 0.55, 100)

profits = [
    calculate_profit(df_val, val_probs < t)
    for t in thresholds
]

best_idx = np.argmax(profits)

best_threshold = thresholds[best_idx]
best_profit = profits[best_idx]

approval_rate = (val_probs < best_threshold).mean()

results = log_run(results, {
    "run_name": "rf_no_balance",
    "model_type": "RandomForestClassifier",
    "class_weight": "None",
    "n_estimators": 200,
    "max_depth": 20,
    "min_samples_leaf": 20,
    "best_threshold": float(best_threshold),
    "auc": float(auc),
    "pr_auc": float(pr_auc),
    "brier": float(brier),
    "profit_at_threshold": float(best_profit),
    "approval_rate": float(approval_rate),
    "train_time_seconds": float(train_time),
    }
)

print(f"\n=== RF (no class_weight) ===")
print(f"Train time:     {train_time:.1f}s")
print(f"AUC:            {auc:.4f}")
print(f"PR-AUC:         {pr_auc:.4f}")
print(f"Brier:          {brier:.4f}")
print(f"Best threshold: {best_threshold:.4f}")
print(f"Approval rate:  {approval_rate:.2%}")
print(f"Profit:         ${best_profit:,.0f}")

rf_b = next(r for r in results if r["run_name"] == "rf_baseline")

print(f"\n--- Comparison ---")
print(f"RF balanced:   AUC {rf_b['auc']:.4f}, Brier {rf_b['brier']:.4f}, Profit ${rf_b['profit_at_threshold']/1e6:.1f}M")
print(f"RF unbalanced: AUC {auc:.4f}, Brier {brier:.4f}, Profit ${best_profit/1e6:.1f}M")

Training RF (no class balancing)...

=== RF (no class_weight) ===
Train time:     45.8s
AUC:            0.7157
PR-AUC:         0.4279
Brier:          0.1609
Best threshold: 0.1409
Approval rate:  35.86%
Profit:         $60,461,642

--- Comparison ---
RF balanced:   AUC 0.7159, Brier 0.2005, Profit $60.5M
RF unbalanced: AUC 0.7157, Brier 0.1609, Profit $60.5M


In [8]:
pd.DataFrame(results).sort_values(
    "profit_at_threshold",
    ascending=False
)

,run_name,model_type,predicted_default_prob,auc,pr_auc,brier,profit_at_threshold,approval_rate,best_threshold,train_time_seconds,excluded_feature,n_estimators,max_depth,min_samples_leaf,class_weight
2,logreg_baseline,LogisticRegression,NaN,0.715654,0.424605,0.219360,60676972.5,0.358256,0.378283,14.756142,NaN,NaN,NaN,NaN,NaN
4,rf_baseline,RandomForestClassifier,NaN,0.715918,0.427309,0.200513,60507642.5,0.362596,0.358081,41.876754,NaN,200.0,20.0,20.0,NaN
5,rf_no_balance,RandomForestClassifier,NaN,0.715690,0.427888,0.160892,60461642.5,0.358587,0.140909,45.805198,NaN,200.0,20.0,20.0,None
3,logreg_no_int_rate,LogisticRegression,NaN,0.714598,0.424545,0.216012,59901460.0,0.344527,0.368182,14.401941,int_rate,NaN,NaN,NaN,NaN
1,baseline_grade_only,grade_lookup,NaN,0.681608,0.359212,0.166423,42848472.5,0.470684,0.125758,NaN,NaN,NaN,NaN,NaN,NaN
0,baseline_approve_all,constant,0.195381,0.500000,0.232832,0.180024,-206712975.0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
with open("../reports/results.json", "w") as f:
    json.dump(results, f, indent=2)